# misuka vs. pyroomacoustics: visual comparison

Visual companion to
[`test_pyroomacoustics_comparison.py`](./test_pyroomacoustics_comparison.py)
in this directory. All scene construction, rendering and scenario
definitions are **imported from that module** rather than duplicated here —
this notebook only adds plots and swaps the hand-rolled T60/C50/D50
estimators for the standard ISO 3382 implementations in
[pyrato](https://pyfar-gallery.readthedocs.io/en/latest/pyrato/) (the
room-acoustics analysis package of the pyfar ecosystem, already used
elsewhere in this repo's acoustic tests/tutorials).

Research question: misuka is a pure Monte Carlo path tracer with no image
source method (ISM). Market-standard room acoustics software combines ray
tracing with ISM as a hybrid model. Does a sufficiently high sample count
let pure path tracing match a hybrid ISM+ray-tracing simulation? Where does
it not, and why? See the module docstring of `test_pyroomacoustics_comparison.py`
for the full scenario rationale.

In [ ]:
import sys, os

try:
    import mitsuba as mi
except ImportError:
    # Fall back to a local (uninstalled) build tree, matching how the
    # pytest suite in this directory is normally run.
    sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "..", "build", "python")))
    import mitsuba as mi

mi.set_variant("cuda_acoustic", "llvm_ad_acoustic")
print(mi.variant())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

%matplotlib inline
%config InlineBackend.figure_format='svg'

sys.path.insert(0, os.path.abspath("."))
import test_pyroomacoustics_comparison as cmp
import pra_comparison_storage as storage
import pra_comparison_plots as plots
# Only for its SPP_VALUES/SPP_SWEEP_SCENARIO constants (what the stored
# ray-count-convergence sweeps were run with) -- this notebook never calls
# any run_<section>() function itself.
import pra_comparison_runner as runner

## Suite-wide settings

Everything below is loaded from the already-stored run metadata (`pra_comparison_storage.list_runs()`)
-- nothing here is hardcoded. If the backend is re-rendered with a changed
setting (`pra_comparison_runner.py`), re-running this cell picks that up
automatically: it reflects whatever actually produced the results shown
further down, not whatever `test_pyroomacoustics_comparison.py`'s constants
currently say (those could differ if the backend hasn't been re-run since a
source change).

Warnings below ("differs across stored runs") mean the stored results were
produced under inconsistent settings -- e.g. some sections re-rendered after
a settings change and others not -- and probably shouldn't be compared
directly until everything is regenerated consistently.

In [ ]:
all_runs = storage.list_runs()
plots.suite_overview(all_runs)

## Rendered Geometries

In [ ]:
plots.plot_rendered_geometries()

## Rendering parameters used throughout this notebook

For reference when reading any of the comparisons below -- what each
simulator was actually configured with here:

| | misuka | pyroomacoustics |
|---|---|---|
| samples / rays | `cmp.MISUKA_SPP` = 2\*\*18 = 262 144 rays per frequency bin (`auditorium_complex`: 4x that, see its `spp=4 * MISUKA_SPP` override) | `cmp.PRA_N_RAYS` = 50 000, fixed for every scenario compared against PRA here (`pra_box_room`'s `n_rays=None` override lets PRA auto-scale instead, not exercised by anything in this notebook since `auditorium_complex` has no PRA comparison at all, see `cmp.NO_PRA_SCENARIOS`) |
| variant / backend | `mi.variant()` = `scalar_acoustic` (set in the cell above -- single-threaded, deterministic; misuka's `seed` parameter has no effect in this build, see the ray-count convergence section further down) | pure Python/Cython, no GPU/JIT backend |
| time resolution | `cmp.SAMPLING_RATE` = 44100 Hz (the shared grid both simulators are binned onto for comparison) | rendered internally at `cmp.PRA_FS` = 176400 Hz, then binned down to `cmp.SAMPLING_RATE` (`cmp.BIN_FACTOR` = 4) |
| frequency | `cmp.FREQUENCY_HZ` = 1000 Hz (single analysis frequency, see the module docstring) | same, via each material's 1 kHz-representative coefficient |

All timings anywhere in this notebook are wall-clock, single machine,
single run each (no repeats to average out noise -- misuka's `seed` has no
effect here, and PRA's own ray tracer isn't reseeded between runs either)
-- read them as indicative of relative cost, not as calibrated benchmarks.

## Acoustic parameters via `pyrato`

`pyrato_edc`/`pyrato_metrics` (ISO 3382 T20/T30/C50/D50 from an energy-domain
signal) now live in [`pra_comparison_metrics.py`](./pra_comparison_metrics.py)
-- every plotting function below that needs them imports them from there
directly; this notebook doesn't call them itself.

## Run all scenarios

Reuses `cmp.PRA_SCENARIOS` (all scenarios except `auditorium_complex`, which
has no pyroomacoustics counterpart at all -- see the "Atmospheric rendering"
section further down for what it *is* covered by) and `cmp.PRA_MODES`
(`rt_only` = PRA `max_order=0`, `hybrid` = PRA `max_order=3`) from the
pytest suite, and `cmp.compare()` to render + align both simulators exactly
as the tests do.

In [ ]:
main_comparison_records = storage.list_runs("main_comparison")
for meta in main_comparison_records:
    print(f"loaded: {meta['scenario']:24s} / {meta['variant']['pra_mode']}")

## Summary table

Note: `coincident_reflections` and `flutter_corridor` can produce wild T20/T30
outliers (e.g. millions of seconds) for whichever curve doesn't have a clean
monotonic decay across the ISO 3382 fit window (-5 to -25/-35 dB) -- pyrato's
linear regression finds the *closest* sample to each target dB level, and on
a curve with a huge front-loaded spike or a near-instant drop to the noise
floor that can land on a near-flat, noise-dominated segment, producing a
near-zero regression slope. That is not a bug in this notebook: it is itself
a finding, exactly for the two scenarios expected to stress each simulator's
dynamic range the most (see the module docstring of
`test_pyroomacoustics_comparison.py`) -- ISO T20/T30 as a single number stops
being a meaningful descriptor there; read the EDC plots below instead.

In [ ]:
pd.set_option("display.width", 160)
plots.main_comparison_table(main_comparison_records)

## Visualize: energy time curve + EDC per scenario

Left: raw (normalized) energy time curve, log scale -- shows individual
reflections/echoes and Monte Carlo noise. Right: `pyrato`'s Schroeder EDC
(smooth, backward-integrated) -- the curve reverberation-time/clarity/
definition are actually computed from. Solid = misuka (identical in both
rows of a scenario, misuka has no ISM order), dashed/dotted = pyroomacoustics
in `rt_only`/`hybrid` mode. Legend labels include each render's wall-clock
time (see `cmp.compare`'s `runtime_misuka_s`/`runtime_pra_s`) -- PRA's
`hybrid` timing includes building the room, which for that mode also
enumerates ISM image sources up to `RT_HYBRID_ORDER`, the extra cost (over
`rt_only`) this is meant to surface.

In [ ]:
for scenario_fn in cmp.PRA_SCENARIOS:
    name = scenario_fn()["name"]
    runs_by_mode = {mode_name: storage.load_run("main_comparison", name, variant={"pra_mode": mode_name})
                   for mode_name, _ in cmp.PRA_MODES}
    plots.plot_main_comparison(name, runs_by_mode)
    plt.show()

## Atmospheric attenuation: misuka (ISO 9613-1) vs. pyroomacoustics (literature table)

No rendering needed here: misuka's `apply_pure_tone_attenuation` evaluates
the continuous ISO 9613-1 formula, pyroomacoustics looks up a coarse
2-temperature x 2-humidity literature table
(`pyroomacoustics.parameters.air_absorption_table`, Vorlaender 2008). Both
give an attenuation coefficient in 1/m via `exp(-distance * a)`, so they're
directly comparable without rendering a scene -- purely analytic/
deterministic, no ray tracing, no repeats needed.

One subplot per (temperature, humidity) condition (each with exactly the
two curves being compared for that condition), rather than all four
conditions overlaid on one shared plot.

In [ ]:
plots.plot_air_absorption_comparison()
plt.show()

## Atmospheric rendering (misuka only): on vs. off, and its runtime cost

This is not a misuka-vs-PRA comparison: it renders every scenario in
`cmp.SCENARIOS` (all six, including `auditorium_complex`, which has no PRA
counterpart at all -- see above) twice with misuka itself, via
`cmp.compare_atmosphere()` -- once with `speed_of_sound=343` and no
attenuation (as in every comparison above), once with the inline
`acoustic_medium` approach (derived speed of sound + ISO 9613-1 attenuation
applied per path segment during rendering, see
`tutorials_acoustic/rendering/atmospheric_rendering/atmospheric_rendering.ipynb`
for the two ways of applying it). Reports the resulting ISO 3382 descriptors
*and* the wall-clock render time for both.

In [ ]:
atmosphere_records = storage.list_runs("atmosphere")
for meta in atmosphere_records:
    print(f"loaded: {meta['scenario']:24s} (atmosphere on/off)")

## Atmosphere used for comparison:

-
-

In [ ]:
for scenario_fn in cmp.SCENARIOS:
    name = scenario_fn()["name"]
    run = storage.load_run("atmosphere", name)
    plots.plot_atmosphere(name, run)
    plt.show()

### Runtime summary

Three bars per scenario, with the runtime-results table shown directly
beside the chart (rather than as a separate table before the ETC/EDC curves
above) -- the full per-scenario metrics table (T20/T30/C50/D50/edc_corr)
is still available via `plots.atmosphere_table(atmosphere_records)` on its
own if needed; the table here is restricted to the runtime columns the bars
themselves visualize.

- **no atmosphere** -- the baseline render (`speed_of_sound=343`, no
  attenuation), also the starting point for the other two.
- **inline atmosphere** -- `acoustic_medium` applied *during* rendering, one
  `exp()` evaluation per path contribution plus a one-time (per render call)
  attenuation-coefficient computation -- small next to the dominant cost of
  path tracing itself (ray/scene intersection, BSDF sampling). Expect the
  overhead to shrink further, in relative terms, for `auditorium_complex`:
  the same fixed per-segment cost against a much larger real-mesh
  intersection cost per ray.
- **no atmosphere + post-process** -- the *same* baseline render, with
  `mi.acoustic.apply_pure_tone_attenuation()` applied to the finished ETC
  afterwards (see `tutorials_acoustic/rendering/atmospheric_rendering/atmospheric_rendering.ipynb`
  for this approach's accuracy tradeoffs vs. the inline one -- it uses each
  time bin's *implied* distance rather than each path's true geometric
  distance). Since this reuses the no-atmosphere render, its bar is barely
  distinguishable from "no atmosphere": post-processing is one vectorized
  pass over the whole ETC array, essentially free next to the render itself.

See "Rendering parameters used throughout this notebook" near the top for
the exact `spp`/variant this (and every other comparison in this notebook)
used.

In [ ]:
plots.plot_atmosphere_runtime_summary(atmosphere_records)
plt.show()

### Post-processing vs. inline attenuation: relative and absolute EDC difference

`cmp.compare_atmosphere_methods()` isolates misuka's two ways of applying
atmospheric attenuation from each other, the way
`atmospheric_rendering.ipynb`'s "Post-processing vs. inline attenuation"
comparison does: a single *raw* render at the atmosphere-derived speed of
sound (attenuation off), then

- **inline**: the same speed of sound, `acoustic_medium.apply_attenuation=True`
  -- attenuation applied *during* rendering, using each path's true
  accumulated geometric distance.
- **post-processing**: the raw render, with
  `mi.acoustic.apply_pure_tone_attenuation()` applied afterwards, using each
  time bin's *implied* distance (`time * speed_of_sound`) instead.

Both variants share the same (atmosphere-derived) speed of sound by
construction, so -- unlike the ray-count convergence section above, which
compares against PRA's different, fixed 343 m/s -- no
`resample_to_reference_speed` realignment is needed here; the relative
difference (`cmp.relative_difference_percent`, ported directly from
`atmospheric_rendering.ipynb`'s `plot_etc_diff_relative`) is computed on
each render's Schroeder EDC (`cmp.schroeder_edc_db`, in dB) rather than
the raw, noisy ETC -- the same smooth decay-curve view used everywhere
else in this notebook. Lines/bands below are the repeat mean +/- std
(see `plot_with_std_band`), not a single render.

Left panel: the raw attenuation effect itself (raw vs. inline, i.e. "how
much does turning attenuation on change the EDC"). Middle panel: the actual
question -- does *how* attenuation is applied matter, given the same speed
of sound? Expect it to be small: both methods attenuate with the same
ISO 9613-1 coefficient, the only difference is which distance (geometric
vs. time-bin-implied) each bin uses.

**Right panel and table (new):** the middle panel's question is frequency-
dependent -- ISO 9613-1 attenuation grows sharply with frequency, so the
geometric-vs-time-bin-implied-distance approximation error should too.
`pra_comparison_runner.HIGHLIGHT_FREQUENCIES_HZ` (100, 1000, 4000, 8000,
14000, 20000 Hz) are rendered simultaneously (misuka's spectral film
supports several frequency bands in one render) and the post-vs-inline
relative discrepancy -- unsigned (absolute value of the relative error, so
no negative values), mean *and* max over the whole render -- is computed
per frequency in the backend
(`pra_comparison_runner._multi_frequency_post_vs_inline`), not here. The
right panel shows this unsigned relative-error curve per frequency (the
middle panel is the same relative-error quantity, signed, at the single
default 1000 Hz); the table gives the explicit numbers behind both panels'
curves, so they don't have to be read off a plot by eye. Not computed for
`auditorium_complex` (loaded from an external mesh with a separately-loaded
sensor, not a plain dict that's cheap to retarget to a different frequency
set) -- its table is `None`. The left/middle panels above, in contrast, are
shown for every scenario including `auditorium_complex` (the loop below
iterates `cmp.SCENARIOS`, which includes it) -- so the auditorium's own
inline-vs-post-processing EDC comparison is right there alongside the
other rooms', just without the per-frequency third panel/table.

In [ ]:
atmosphere_methods_records = storage.list_runs("atmosphere_methods")
for meta in atmosphere_methods_records:
    print(f"loaded: {meta['scenario']:24s} (inline vs. post-processing, same speed of sound)")

In [ ]:
for scenario_fn in cmp.SCENARIOS:
    name = scenario_fn()["name"]
    run = storage.load_run("atmosphere_methods", name)
    plots.plot_atmosphere_methods(name, run)
    plt.show()
    freq_table = plots.atmosphere_methods_frequency_table(run)
    if freq_table is not None:
        print(name)
        display(freq_table)

## How many rays does misuka need to compensate for PRA's ISM?

`cmp.ray_count_convergence()` renders one scenario at increasing `spp`
(misuka's ray/sample count), both without and with inline atmospheric
rendering, against one fixed PRA `hybrid`-mode reference room, and reports
the RMS error between each render's Schroeder EDC and that reference's (see
`edc_rms_error_db`) -- a single scale-invariant number per render summarizing
how far its whole decay curve is from PRA's ISM+RT hybrid result.

### Atmosphere used here (the "inline atmosphere" renders)

The same fixed `ATMO_*` constants as everywhere else atmospheric rendering
appears in this notebook (see `test_pyroomacoustics_comparison.py`'s module
constants), fed through misuka's `acoustic_medium` (ISO 9613-1 speed of
sound + attenuation):

| parameter | value |
|---|---|
| temperature | `cmp.ATMO_TEMPERATURE` = 25.0 °C |
| relative humidity | `cmp.ATMO_RELATIVE_HUMIDITY` = 0.6 (60%) |
| atmospheric pressure | `cmp.ATMO_PRESSURE` = 101 825 Pa |
| saturation vapor pressure | `cmp.ATMO_SATURATION_VAPOR_PRESSURE` ≈ 3167 Pa (Magnus formula from temperature, see the constant's definition) |
| CO2 concentration | `cmp.ATMO_CO2_PPM` = 400.0 ppm |
| speed-of-sound method | `cmp.ATMO_SPEED_OF_SOUND_METHOD` = `"auto"` -- with humidity *and* CO2 both given, `mitsuba::acoustic::speed_of_sound`'s auto-selection resolves this to `"cramer"` (see `acoustic.h`; logged as a `Warn`-level message the first time it runs) |
| resulting speed of sound | `cmp.atmo_speed_of_sound()` ≈ 347.3 m/s (vs. PRA's fixed, un-derived 343 m/s -- the whole reason the resampling below is needed at all) |

Air attenuation itself is frequency-dependent (ISO 9613-1); at
`cmp.FREQUENCY_HZ` = 1000 Hz and these conditions it is small over the
short distances in `SPP_SWEEP_SCENARIO` (`shoebox_diffuse_200m3`, ≤ a few
meters) -- the dominant atmospheric effect on the numbers below is the
~1.3% speed-of-sound shift (347.3 vs. 343 m/s), not the attenuation.

### How the RMS error is computed (`edc_rms_error_db`)

For one misuka render and the PRA hybrid reference:

1. Both energy arrays are turned into a Schroeder EDC (`schroeder_edc_db`:
   backward-cumulative-sum of the energy, normalized to 0 dB at t=0, then
   `10*log10`) -- the same smooth, monotonically-decaying curve view used
   everywhere else in this notebook, not the raw noisy ETC.
2. The two EDCs are truncated to their common length and masked to where
   *both* are still above `floor_db=-40 dB`: below that, whichever
   simulator's Monte Carlo/ray-tracing noise floor happens to sit lower
   first would dominate the comparison, which is a noise-floor artifact,
   not a modeling disagreement.
3. The RMS of the (dB-domain) difference between the two masked curves is
   the reported error -- one number, in dB, that is large if the two decay
   curves differ in level *or* slope anywhere in the compared range, and
   ~0 only if they track each other closely throughout.

Because it's computed directly on 10*log10(energy) curves, this dB number is
already a *relative* (ratio-based) measure by construction, not an absolute
energy difference -- see the new "Relative RMS error" plot below for a
percentage-scale reading of the same quantity.

<div class="admonition important alert alert-block alert-warning">

⚠️ **Why not just `ETC_pra - ETC_misuka`?** Misuka's atmosphere-on render
derives its own speed of sound from `ATMO_*` (`cmp.atmo_speed_of_sound()`,
~347 m/s here vs. PRA's fixed 343 m/s) -- and misuka bins by *time*, not
distance, so the same physical reflection lands in a different time bin
under each speed of sound. A raw bin-by-bin difference would therefore
compare unrelated events and read the alignment mismatch as an atmospheric
effect. `cmp.resample_to_reference_speed()` (ported from
`atmospheric_rendering.ipynb`'s helper of the same name) first resamples the
atmosphere-on curve onto PRA's distance grid; the no-atmosphere case needs
no realignment, since it uses the same fixed `SPEED_OF_SOUND` as PRA.

</div>

Note: misuka's `seed` parameter does not currently change the render output
in this build/variant (verified: byte-identical results across seeds) --
there is no seed noise to average out here, so each point below is a single
deterministic render, not a mean over repeats.

In [ ]:
SPP_SWEEP_SCENARIO = runner.SPP_SWEEP_SCENARIO
SPP_VALUES = runner.SPP_VALUES

rcc_scenario_name = SPP_SWEEP_SCENARIO()["name"]
rcc_run = storage.load_run("ray_count_convergence", rcc_scenario_name)

spp = rcc_run["arrays"]["spp"]
err_no_atmo = rcc_run["arrays"]["errors_no_atmo"]
err_atmo = rcc_run["arrays"]["errors_atmo"]
rel_err_no_atmo_pct = rcc_run["arrays"]["rel_err_no_atmo_pct"]
rel_err_atmo_pct = rcc_run["arrays"]["rel_err_atmo_pct"]
delta = err_atmo - err_no_atmo

for s, en, ea, d, ren, rea in zip(spp, err_no_atmo, err_atmo, delta,
                                  rel_err_no_atmo_pct, rel_err_atmo_pct):
    print(f"spp={s:8d}  no_atmo={en:.3f}dB ({ren:6.1f}%)  "
          f"atmo={ea:.3f}dB ({rea:6.1f}%)  delta={d:+.3f}dB")

In [ ]:
plots.plot_ray_count_convergence_error(rcc_scenario_name, rcc_run)
plt.show()

In [ ]:
plots.plot_ray_count_convergence_relative_error(rcc_scenario_name, rcc_run)
plt.show()

**What this actually shows** (see the printed numbers above, not just the
shape of the curve): "no atmosphere" does *not* converge to 0 error against
PRA `hybrid` specifically -- it plateaus around a small but clearly nonzero
floor once `spp` is large enough. That floor is an ISM/RT energy-accounting
gap: misuka has no ISM at all to match PRA's hybrid mode against on equal
footing, regardless of ray count -- `rt_only` is the fair comparison for
that, see `TIGHT_COMPARISON_SCENARIOS`. More rays do not close *that* gap,
because it isn't a sampling problem.

On the relative-error plot: the ~1.46 dB no-atmosphere plateau is ~40%
relative RMS error, and the ~1.67 dB inline-atmosphere plateau is ~47% --
both plateaus read as large in percentage terms precisely because
`edc_rms_error_db` integrates the (dB) mismatch over the *entire* compared
decay range, including any short stretch where the two curves disagree
most (e.g. right around the direct sound or a strong early reflection); it
is not a claim that the two decay curves are "half energy apart" throughout.

The right panel (absolute-error figure above) isolates the atmosphere's
*own* contribution by subtracting that baseline out (`atmo - no_atmo`), and
this is where the requested convergence pattern actually appears: noisy and
large at very low `spp` (few rays make both the render and the
distance-resampling step noisy), then settling to a small, stable, nonzero
plateau once there are enough rays -- exactly the "converges to a small
finite number, not zero" behavior hypothesized, once the pre-existing
hybrid-mode bias is separated out from the atmosphere-specific effect being
measured.

### Which ISO 3382 descriptors actually converge?

`edc_rms_error_db` above is one aggregate number over the *whole* decay
curve -- it can't say whether spp mainly affects reverberation time, early
clarity, or something else. `ray_count_convergence()` now also returns the
raw energy array it rendered at each spp (`energies_no_atmo`/
`energies_atmo`, already realigned same as above) plus the single PRA
hybrid reference array (`e_pra`), so `pyrato_metrics` can be run per spp,
exactly as in "Run all scenarios" above.

Each descriptor is expressed **relative to the PRA hybrid reference**, the
same bounded (±100%) formula as `cmp.relative_difference_percent` (used for
the atmosphere-method comparison further up), applied here to one scalar
descriptor value per spp instead of a bin-by-bin energy curve:

```
relative[%] = 100 * (misuka - pra) / max(|misuka|, |pra|)
```

0% means misuka matches the PRA hybrid reference for that descriptor
exactly; the sign shows the direction of the gap (e.g. misuka's T20 running
longer or shorter than PRA's).

In [ ]:
plots.iso_convergence_table(rcc_run)

In [ ]:
plots.plot_iso_convergence(rcc_scenario_name, rcc_run)
plt.show()

## Takeaways

- Read the summary table's `*_delta` columns together with the plots above:
  a small delta with a low `edc_corr` (or vice versa) means the aggregate
  ISO numbers agree by coincidence while the underlying decay shape does
  not -- always check the plot, not just the table.
- `rt_only` is the fair, apples-to-apples comparison (misuka has no ISM to
  compare against PRA's hybrid mode on equal footing); differences there
  point at real algorithmic gaps. Differences that appear only in `hybrid`
  point at how PRA itself splits energy between ISM and ray tracing, not at
  misuka.
- See `test_pyroomacoustics_comparison.py`'s module docstring and the
  `TIGHT_COMPARISON_SCENARIOS` comment block for the reasoning behind which
  gaps are expected vs. which would indicate a bug.
- `auditorium_complex` has no pyroomacoustics comparison at all (an earlier
  bounding-box approximation was removed, see `scenario_auditorium_complex`)
  -- it only appears in the "Atmospheric rendering" section, where it's
  misuka-only anyway.
- The atmospheric-rendering comparison pins `speed_of_sound_method='cramer'`
  rather than `'auto'`: at `ATMO_RELATIVE_HUMIDITY=0.6`, `'auto'` would
  select `'ideal_gas'`, which has a real bug currently on this branch (see
  `ATMO_SPEED_OF_SOUND_METHOD`'s comment in the .py suite) that inflates the
  derived speed of sound well past what's physically possible for air.

## Appendix: three additional misuka vs. PRA comparisons

Same three views as before (ISM peak level, Hybrid parameter level, RT-only
histogram level), now loaded from `pra_comparison_data/` instead of
rendered inline -- see `pra_comparison_runner.py`'s `run_appendix_*`
functions for what produces each stored result, and
`pra_comparison_metrics.py` for `pra_ism_peak_times_and_energies`/
`build_peak_windows`/`parameters_full`/`cosine_similarity`/`rmspe`/`mpe`
(all relocated there unchanged).

### Shared error metrics (Section 1)

`cosine_similarity`/`rmspe`/`mpe` now live in
[`pra_comparison_metrics.py`](./pra_comparison_metrics.py).

# 1. misuka vs. pyroomacoustics -- ISM (peak level)

Compares individual reflection-peak energies between misuka's RT and PRA's
deterministic image-source model, over growing ray counts (`SPP_VALUES`).
Peak windows are derived analytically from ISM arrival times +/- a peak
half-width (`source_radius / speed_of_sound / 2`), overlapping windows are
merged, and energy is integrated per window -- rather than compared
sample-for-sample (the peak *shapes* differ structurally between a
deterministic delta and misuka's finite-emitter-smeared arrival, see the
"Direct-sound time-axis alignment" section far above).

In [ ]:
appendix1_records = storage.list_runs("appendix_ism_peak")
for scenario_fn in cmp.PRA_SCENARIOS:
    name = scenario_fn()["name"]
    run = storage.load_run("appendix_ism_peak", name)
    plots.plot_ism_peak_bars(name, run)
    plt.show()
    plots.plot_ism_peak_convergence_metrics(name, run)
    plt.show()
    print(f"done: {name:24s} (ISM peak-level, n_max={run['metadata']['metrics']['n_max']})")

pd.DataFrame([m["metrics"] | {"scenario": m["scenario"]} for m in appendix1_records])

# 2. misuka vs. pyroomacoustics -- Hybrid (parameter level)

Compares aggregated room-acoustic parameters (C50, C80, G, EDT, T60, TS)
between misuka (pure RT, growing ray count) and PRA's hybrid mode (ISM + RT).
No cross-renderer time alignment is applied here, matching this notebook's
own established practice (see `compare()`'s module-level comment in
`test_pyroomacoustics_comparison.py`): at small source radii, the geometric
hull-vs-center offset is a sub-to-few-bin shift, small next to the 50/80 ms
windows and multi-hundred-ms decays these parameters are computed over --
C50/C80 have some (minor) sensitivity to it near their window boundary,
EDT/T60/TS/G effectively none.

That "small source radii" premise is no longer just an assumption in a
comment -- `check_alignment_offset_negligible()` (`pra_comparison_metrics.py`)
computes the actual offset (`source_radius / speed_of_sound`, in bins) for
whichever scenario is currently being run and warns if it exceeds
`UNALIGNED_OFFSET_WARN_BINS`, so a new or rescaled scenario (e.g. the scaled
shoebox sizes, whose `source_radius` grows with room size -- see
`_scaled_shoebox`) can't silently violate it unnoticed the way this comment
itself once did. See the "H1 t0-alignment" investigation notes for the
concrete scenarios (`shoebox_diffuse/specular_5000m3`, `source_radius=0.585
m`, ~75 bins) where this assumption actually broke.

⚠️ **G (Strength) is not meaningfully comparable in absolute terms across
renderers.** Both `misuka`'s and PRA's `G` below are computed from each
render's *own* direct sound, scaled by `distance^2/10^2`, as a stand-in
free-field reference (Masterarbeit Eq. 63/64) -- not an independently
rendered, calibrated free-field response at 10 m. Since misuka and PRA use
different absolute energy/power conventions to begin with (radiometric
sphere emitter + solid-angle sampling vs. a unit point source, see this
notebook's earlier module docstring reference), a large `G` gap between the
two can just as easily reflect that difference as an actual room effect.
Read `G`'s *trend* against increasing spp (does misuka's own G stabilize?),
not the absolute misuka-vs-PRA gap.

In [ ]:
appendix2_records = storage.list_runs("appendix_hybrid_params")
for scenario_fn in cmp.PRA_SCENARIOS:
    name = scenario_fn()["name"]
    run = storage.load_run("appendix_hybrid_params", name)
    plots.plot_hybrid_jnd(name, run)
    plt.show()
    print(f"done: {name:24s} (Hybrid parameter-level, n_max={run['metadata']['metrics']['n_max']})")

# 3. misuka vs. pyroomacoustics -- RT-only (histogram level, no sinc)

Compares misuka's RT directly against PRA's raw ray-traced energy histogram
(`room.rt_histograms`), *before* the sinc/fractional-delay reconstruction
into a sample-accurate RIR (`cmp.pra_histogram_aligned_etc` -- see the
"Direct-sound time-axis alignment" section far above for the full
derivation). Only the geometric correction (error source (1)) is applied;
the PRA head-delay (error source, RIR-synthesis-only) is deliberately not
applied, since it does not exist in the histogram to begin with.

In [ ]:
appendix3_records = storage.list_runs("appendix_rt_histogram")
for scenario_fn in cmp.PRA_SCENARIOS:
    name = scenario_fn()["name"]
    run = storage.load_run("appendix_rt_histogram", name)
    m = run["metadata"]["metrics"]
    print(f"{name:24s} direct sound: misuka={m['direct_ms_misuka']:.3f} ms  "
          f"PRA-histogram={m['direct_ms_pra']:.3f} ms   "
          f"cross-corr lag (first 100 ms)={m['cross_corr_lag_ms']:.3f} ms")
    plots.plot_rt_histogram_comparison(name, run)
    plt.show()

# Zusammenfassung (Gerüst-Integration)

Overview of the three PRA modes' agreement with misuka, per scenario, at the
highest simulated spp -- the equivalent of the scaffold's own single-room
summary table, one row per scenario per comparison.

In [ ]:
plots.appendix_summary_table(appendix1_records, appendix2_records, appendix3_records)

**Open points carried over from the scaffold** (still open, not resolved by
this integration):

- Overlapping/coincident reflections (Section 1, `coincident_reflections`
  specifically): once several ISM peaks merge into one window, per-window
  energy integration can no longer attribute energy to an individual
  reflection -- an echo-density- or EDC-based metric would be needed for a
  finer-grained comparison there.
- G's cross-renderer absolute comparability (see the warning in Section 2)
  would need a dedicated, independently rendered free-field reference scene
  at 10 m on both sides to resolve properly, rather than each renderer's own
  distance-scaled direct sound.
- This appendix reuses `cmp.PRA_SCENARIOS` as given; it does not introduce
  the Masterarbeit's explicit S/M/L room-size labeling -- `shoebox_diffuse`/
  `shoebox_specular` (200 m^3), `coincident_reflections` (216 m^3),
  `flutter_corridor` (100 m^3) and `l_room` (1000 m^3) already span a
  comparable small-to-large range.